# Construction actions: d3-nano vs LFM3-VL-2B

Run the setup cells once on an L4. Then rerun **Upload**, **Inference**, and **Results** whenever you want to test another 1–3 videos. Both models stay loaded in GPU memory.

Labels: **standing still**, **driving**, **digging**, **other**. Scores are relative to the given options and are not calibrated accuracy probabilities.

## 1. Install

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.21.0', 'accelerate', 'safetensors', 'einops',
    'opencv-python-headless==4.11.0.86'], check=True)


## 2. Imports and shared question

In [ ]:
import json, time
from pathlib import Path
import cv2
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from transformers import AutoModel, AutoProcessor, AutoModelForImageTextToText

assert torch.cuda.is_available(), 'Select a GPU in Runtime > Change runtime type.'
print('GPU:', torch.cuda.get_device_name(0))

LABELS = {
    'standing_still': 'Vehicle and working attachment remain still, ignoring minor vibration or foliage.',
    'driving': 'The vehicle travels across the site; moving the attachment alone is not driving.',
    'digging': 'The tool excavates, scrapes or scoops soil or material.',
    'other': 'Unclear, mixed, or another action, including tool movement without clear digging.'
}
QUESTION = 'Which option best describes the dominant observable activity in this silent construction-equipment POV video?'


## 3. Download and load d3-nano

In [ ]:
start = time.perf_counter()
d3 = AutoModel.from_pretrained(
    'vllm-sr/d3-nano', revision='8198c9f763b046605b2f4827b4999971fa2c92c4',
    trust_remote_code=True, device='cuda:0', batch_size=1)
torch.cuda.synchronize()
print(f'd3 ready in {time.perf_counter() - start:.1f}s')


## 4. Download and load LFM3-VL-2B

In [ ]:
start = time.perf_counter()
lfm_id = 'LiquidAI/LFM3-VL-2B'
lfm_revision = '71d8e745d97c224984e0f20412f532a5ce9dfd3e'
processor = AutoProcessor.from_pretrained(lfm_id, revision=lfm_revision, trust_remote_code=True)
lfm = AutoModelForImageTextToText.from_pretrained(
    lfm_id, revision=lfm_revision, trust_remote_code=True,
    dtype=torch.bfloat16, device_map={'': 'cuda:0'}, attn_implementation='sdpa').eval()
processor.tokenizer.padding_side = 'left'
letters = list('ABCD')
token_ids = [processor.tokenizer.encode(letter, add_special_tokens=False) for letter in letters]
assert all(len(ids) == 1 for ids in token_ids), 'Expected one token per answer letter.'
torch.cuda.synchronize()
print(f'LFM ready in {time.perf_counter() - start:.1f}s')


## 5. Upload 1–3 short videos
Rerun from this cell to test more videos without downloading or loading the models again.

In [ ]:
uploaded = files.upload()
assert 1 <= len(uploaded) <= 3, 'Upload 1–3 videos.'
video_paths = []
for i, (name, content) in enumerate(uploaded.items()):
    path = Path(f'/content/action_video_{i}.mp4')
    path.write_bytes(content)
    video_paths.append((name, path))
del uploaded
print('Videos:', [name for name, _ in video_paths])


## 6. Classify videos
Both models remain loaded. d3 processes independent videos one at a time; its video list represents a **single combined request**. LFM processes the uploaded videos in one batch. Times include video preparation and inference, but exclude downloads and model loading.

In [ ]:
def sample_video(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or fps <= 0 or total < 4:
        cap.release()
        raise ValueError(f'Invalid or too-short video: {path}')
    indices = np.linspace(0, total - 1, min(max(int(total / fps * 2), 4), 32, total)).round().astype(int).tolist()
    wanted = set(indices)
    frames = []
    index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if index in wanted:
            h, w = frame.shape[:2]
            scale = min(1.0, 512 / max(h, w))
            frame = cv2.resize(frame, (round(w * scale), round(h * scale)))
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        index += 1
    cap.release()
    if len(frames) != len(indices):
        raise ValueError(f'Could not decode all sampled frames: {path}')
    if len(frames) % 2:
        frames.append(frames[-1])
        indices.append(indices[-1])
    metadata = {'fps': fps, 'total_num_frames': total, 'duration': total / fps,
                'frames_indices': indices, 'width': frames[0].shape[1], 'height': frames[0].shape[0]}
    return np.stack(frames), metadata

results = []
d3_total = 0.0
for name, path in video_paths:
    torch.cuda.synchronize()
    start = time.perf_counter()
    response = d3.system_one(state=QUESTION, videos=[str(path)], questions={
        'activity': {'type': 'choice', 'instructions': QUESTION, 'criteria': LABELS}})
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    answer = response['answers']['activity']
    if 'probabilities' not in answer:
        raise RuntimeError(f'd3 returned an error for {name}: {answer}')
    scores = {k: float(v) for k, v in answer['probabilities'].items()}
    results.append({'video': name, 'model': 'd3', 'label': max(scores, key=scores.get),
                    'option_scores': scores, 'seconds': elapsed, 'timing': 'individual'})
    d3_total += elapsed

# One LFM forward pass for the full set of independent clips.
torch.cuda.synchronize()
start = time.perf_counter()
prepared = [sample_video(path) for _, path in video_paths]
options = '\n'.join(f'{letter}. {key}: {LABELS[key]}' for letter, key in zip(letters, LABELS))
prompt = QUESTION + '\n' + options + '\nAnswer with exactly one letter A, B, C, or D.'
texts = [processor.apply_chat_template(
    [{'role': 'user', 'content': [{'type': 'video'}, {'type': 'text', 'text': prompt}]}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False) for _ in prepared]
inputs = processor(text=texts, videos=[p[0] for p in prepared], video_metadata=[p[1] for p in prepared],
                   do_sample_frames=False, padding=True, return_tensors='pt')
inputs = {k: v.to('cuda:0') if hasattr(v, 'to') else v for k, v in inputs.items() if k != 'video_metadata'}
with torch.inference_mode():
    logits = lfm(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1]
    probabilities = torch.softmax(logits[:, [ids[0] for ids in token_ids]].float(), dim=-1).cpu().numpy()
torch.cuda.synchronize()
lfm_total = time.perf_counter() - start
for (name, _), probabilities_row in zip(video_paths, probabilities):
    scores = dict(zip(LABELS, map(float, probabilities_row)))
    results.append({'video': name, 'model': 'lfm', 'label': max(scores, key=scores.get),
                    'option_scores': scores, 'seconds': lfm_total / len(video_paths),
                    'timing': 'batch average'})
print(f'd3: {d3_total:.2f}s total | LFM: {lfm_total:.2f}s total (one batch)')


## 7. Results
For LFM, per-video time is the batch total divided by the number of videos; it is not a separately measured duration. The first inference may include GPU warm-up.

In [ ]:
table = pd.DataFrame([{k: row[k] for k in ['video','model','label','seconds','timing']} for row in results])
display(table)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for model in ['d3', 'lfm']:
    rows = [r for r in results if r['model'] == model]
    axes[0].plot([r['video'] for r in rows], [list(LABELS).index(r['label']) for r in rows], 'o-', label=model)
    axes[1].plot([r['video'] for r in rows], [r['seconds'] for r in rows], 'o-', label=model)
axes[0].set_yticks(range(len(LABELS)), list(LABELS))
axes[0].set_title('Decision per video')
axes[1].set_title('Processing time per video (LFM: batch average)')
axes[1].set_ylabel('Seconds')
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
    ax.legend()
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()
print(json.dumps(results, indent=2))
output = Path('/content/action_results.json')
output.write_text(json.dumps({'results': results, 'd3_total_seconds': d3_total, 'lfm_batch_seconds': lfm_total}, indent=2))
files.download(str(output))
